# Prediksi Risiko Putus Kuliah Mahasiswa Menggunakan Random Forest

**Mata kuliah:** Pembelajaran Mesin &nbsp;|&nbsp; **Tugas:** UTS (sekaligus bahan TA-06)
**Dataset:** *Predict Students' Dropout and Academic Success* (UCI, ID 697) &nbsp;|&nbsp; **Algoritma:** Random Forest

Notebook ini dapat dijalankan di **Google Colab** maupun **VS Code** (letakkan di folder `notebook/` proyek).
Semua keluaran (model, grafik, metrik) otomatis disimpan ke folder yang dipakai aplikasi web Flask:

| Keluaran | Lokasi |
|---|---|
| Model | `model/rf_model.joblib` |
| Daftar fitur, label, metrik, contoh data | `model/*.json` |
| Grafik | `static/charts/*.png` |
| Daftar library | `requirements.txt` |

**Alur kerja:** muat data → eksplorasi → split 80:20 → Skenario A (jumlah pohon) → Skenario B (kedalaman pohon) →
Skenario C (bobot kelas) → Skenario D (prediksi dini: kelompok fitur) → feature importance → seleksi fitur untuk web →
model akhir → simpan.

## 1. Persiapan

In [ ]:
import json, sys, shutil, warnings, zipfile, platform
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
IN_COLAB = "google.colab" in sys.modules

# --- Lokasi folder: jika notebook ada di folder "notebook/", proyek = folder induknya ---
CWD = Path.cwd()
BASE_DIR = CWD.parent if CWD.name == "notebook" else CWD
DATA_DIR, MODEL_DIR = BASE_DIR / "data", BASE_DIR / "model"
CHART_DIR = BASE_DIR / "static" / "charts"
for d in (DATA_DIR, MODEL_DIR, CHART_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Gaya grafik (selaras dengan tema web hijau pastel) ---
HIJAU_TUA, HIJAU, HIJAU_PASTEL = "#2B3F33", "#4F7A5C", "#A9C5A0"
KREM, KREM_TUA, LATAR = "#D8C9A8", "#B49B6A", "#F4F7EF"
WARNA_KELAS = {"Dropout": "#D9A99A", "Enrolled": "#D8C9A8", "Graduate": "#8FB58A"}
plt.rcParams.update({
    "figure.facecolor": LATAR, "axes.facecolor": LATAR, "savefig.facecolor": LATAR,
    "axes.edgecolor": HIJAU_TUA, "axes.labelcolor": HIJAU_TUA, "text.color": HIJAU_TUA,
    "xtick.color": HIJAU_TUA, "ytick.color": HIJAU_TUA, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": "#D5E0CD",
    "axes.axisbelow": True, "font.size": 11, "axes.titleweight": "bold",
})
CMAP_HIJAU = LinearSegmentedColormap.from_list("hijau", [LATAR, HIJAU_PASTEL, HIJAU])

def simpan_grafik(nama):
    plt.tight_layout()
    plt.savefig(CHART_DIR / f"{nama}.png", dpi=150)
    plt.show()

def tulis_json(path, obj):
    def ke_python(o):
        return o.item() if hasattr(o, "item") else str(o)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=ke_python)

print("Folder proyek :", BASE_DIR)
print("scikit-learn  :", sklearn.__version__, "| pandas:", pd.__version__, "| Python:", platform.python_version())

## 2. Memuat Dataset

File asli dari UCI memakai pemisah **titik koma (`;`)**, diawali karakter BOM, dan ada nama kolom yang berakhiran karakter *tab*
(`Daytime/evening attendance`). Ketiganya ditangani di sel berikut.

> Letakkan `data.csv` di folder `data/`. Di Colab, jika file belum ada, Anda akan diminta mengunggah `data.csv` (atau ZIP dari UCI).

In [ ]:
DATA_PATH = DATA_DIR / "data.csv"

if not DATA_PATH.exists():
    if IN_COLAB:
        from google.colab import files
        print("Unggah file data.csv (atau ZIP dari UCI)...")
        unggahan = files.upload()
        nama_file = next(iter(unggahan))
        if nama_file.lower().endswith(".zip"):
            with zipfile.ZipFile(nama_file) as z:
                DATA_PATH.write_bytes(z.read("data.csv"))
        else:
            DATA_PATH.write_bytes(unggahan[nama_file])
    else:
        raise FileNotFoundError(f"Taruh data.csv di {DATA_DIR}")

df = pd.read_csv(DATA_PATH, sep=";", encoding="utf-8-sig")
df.columns = df.columns.str.strip()          # buang tab/spasi di nama kolom
print("Ukuran data:", df.shape)
display(df.head())

## 3. Eksplorasi Data (EDA)

In [ ]:
print("Jumlah baris, kolom :", df.shape)
print("Total missing value :", int(df.isna().sum().sum()))
print("Baris duplikat      :", int(df.duplicated().sum()))
print("Tipe data           :", df.dtypes.value_counts().to_dict())

dist = df["Target"].value_counts()
persen = dist / dist.sum() * 100
fig, ax = plt.subplots(figsize=(6.5, 4.2))
batang = ax.bar(dist.index, dist.values, color=[WARNA_KELAS[k] for k in dist.index], edgecolor=HIJAU_TUA)
for b, p in zip(batang, persen):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 25, f"{int(b.get_height())}\n({p:.1f}%)", ha="center")
ax.set_title("Distribusi Kelas Target")
ax.set_ylabel("Jumlah mahasiswa")
ax.set_ylim(0, dist.max() * 1.18)
ax.grid(axis="x", visible=False)
simpan_grafik("distribusi_kelas")
display(df.describe().T.round(2))

## 4. Preprocessing dan Pembagian Data

* Target diubah ke angka (urutan abjad: `Dropout=0`, `Enrolled=1`, `Graduate=2`).
* Semua fitur sudah numerik (kategori sudah berupa **kode angka** dan tidak ada *missing value*), sehingga tidak diperlukan *one-hot encoding*.
  Pohon keputusan membuat pemisahan berbasis ambang batas, sehingga tidak mengandaikan hubungan linear antar kode.
* Pembagian **80% latih : 20% uji** dengan `stratify` agar proporsi kelas terjaga.
* Kelas **tidak seimbang** (Graduate terbanyak, Enrolled tersedikit), sehingga metrik utama yang dipakai adalah **F1-macro**, bukan hanya akurasi.

In [ ]:
TARGET = "Target"
KELAS = sorted(df[TARGET].unique())                     # ['Dropout', 'Enrolled', 'Graduate']
KE_ANGKA = {k: i for i, k in enumerate(KELAS)}
label_map = {i: k for k, i in KE_ANGKA.items()}
y = df[TARGET].map(KE_ANGKA)
X = df.drop(columns=[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Data latih :", X_train.shape, "| Data uji:", X_test.shape)
print("\nProporsi kelas (%):")
display(pd.DataFrame({
    "Seluruh data": y.map(label_map).value_counts(normalize=True) * 100,
    "Latih": y_train.map(label_map).value_counts(normalize=True) * 100,
    "Uji": y_test.map(label_map).value_counts(normalize=True) * 100,
}).round(1))

## 5. Fungsi Bantu dan Model Dasar (Baseline)

Pemilihan parameter terbaik memakai **validasi silang 5-lipatan pada data latih** (F1-macro), bukan data uji,
supaya data uji tetap "bersih" untuk evaluasi akhir.

In [ ]:
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def buat_model(**kw):
    return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **kw)

def jalankan(kw, Xtr, Xte):
    """Latih model, hitung metrik training/testing dan F1-macro validasi silang."""
    model = buat_model(**kw).fit(Xtr, y_train)
    pred = model.predict(Xte)
    lap = classification_report(y_test, pred, target_names=KELAS, output_dict=True, zero_division=0)
    hasil = {
        "acc_train": accuracy_score(y_train, model.predict(Xtr)),
        "acc_test": accuracy_score(y_test, pred),
        "f1_test": f1_score(y_test, pred, average="macro"),
        "recall_enrolled": lap["Enrolled"]["recall"],
        "recall_dropout": lap["Dropout"]["recall"],
        "cv_f1": cross_val_score(buat_model(**kw), Xtr, y_train, cv=SKF, scoring="f1_macro", n_jobs=-1).mean(),
    }
    return hasil, model

dasar, _ = jalankan({"n_estimators": 100}, X_train, X_test)
print("Model dasar (100 pohon, kedalaman bebas):")
print({k: round(v, 4) for k, v in dasar.items()})

## 6. Skenario A: Pengaruh Jumlah Pohon (`n_estimators`)
Variasi: 10, 50, 100, 200, 500. Pertanyaan analisis: *apakah menambah pohon terus-menerus meningkatkan akurasi secara signifikan?*

In [ ]:
JUMLAH_POHON = [10, 50, 100, 200, 500]
baris = []
for n in JUMLAH_POHON:
    h, _ = jalankan({"n_estimators": n}, X_train, X_test)
    baris.append({"n_estimators": n, **h})
    print(f"n_estimators={n:<4} acc_test={h['acc_test']:.4f}  cv_f1={h['cv_f1']:.4f}")
df_a = pd.DataFrame(baris)
display(df_a.round(4))

fig, ax = plt.subplots(figsize=(7.2, 4.2))
pos = list(range(len(df_a)))
ax.plot(pos, df_a["acc_test"], marker="o", color=HIJAU, lw=2.2, label="Akurasi testing")
ax.plot(pos, df_a["cv_f1"], marker="^", color=KREM_TUA, lw=2.2, ls="--", label="F1-macro validasi silang (CV)")
ax.set_xticks(pos); ax.set_xticklabels(df_a["n_estimators"])
ax.set_xlabel("Jumlah pohon (n_estimators)"); ax.set_ylabel("Skor")
ax.set_title("Skenario A: Pengaruh Jumlah Pohon"); ax.legend()
simpan_grafik("skenario_a")

N_TERBAIK = int(df_a.sort_values(["cv_f1", "n_estimators"], ascending=[False, True]).iloc[0]["n_estimators"])
selisih = df_a["acc_test"].max() - df_a["acc_test"].min()
print(f"n_estimators terbaik (CV F1-macro): {N_TERBAIK}")
print(f"Rentang akurasi testing antar pengaturan: {selisih:.4f} ({selisih*100:.2f} poin persentase)")

## 7. Skenario B: Pengaruh Kedalaman Pohon (`max_depth`)
Memakai `n_estimators` terbaik dari Skenario A. Variasi: 3, 5, 10, 20, `None` (tanpa batas).
Bandingkan akurasi **training** dan **testing**: selisih yang besar menandakan *overfitting*.

In [ ]:
KEDALAMAN = [3, 5, 10, 20, None]
baris = []
for d in KEDALAMAN:
    h, _ = jalankan({"n_estimators": N_TERBAIK, "max_depth": d}, X_train, X_test)
    baris.append({"max_depth": "Tanpa batas" if d is None else d, "_nilai": 999 if d is None else d, **h})
    print(f"max_depth={str(d):<5} train={h['acc_train']:.4f}  test={h['acc_test']:.4f}  cv_f1={h['cv_f1']:.4f}")
df_b = pd.DataFrame(baris)
df_b["selisih_train_test"] = df_b["acc_train"] - df_b["acc_test"]
display(df_b.drop(columns="_nilai").round(4))

fig, ax = plt.subplots(figsize=(7.2, 4.2))
pos = list(range(len(df_b)))
ax.plot(pos, df_b["acc_train"], marker="s", color=KREM_TUA, lw=2.2, ls="--", label="Akurasi training")
ax.plot(pos, df_b["acc_test"], marker="o", color=HIJAU, lw=2.2, label="Akurasi testing")
ax.set_xticks(pos); ax.set_xticklabels(df_b["max_depth"])
ax.set_xlabel("Kedalaman maksimum pohon (max_depth)"); ax.set_ylabel("Akurasi")
ax.set_title("Skenario B: Pengaruh Kedalaman Pohon"); ax.legend()
simpan_grafik("skenario_b")

terbaik_b = df_b.sort_values(["cv_f1", "_nilai"], ascending=[False, True]).iloc[0]
DEPTH_TERBAIK = None if terbaik_b["max_depth"] == "Tanpa batas" else int(terbaik_b["max_depth"])
awal_overfit = df_b[df_b["selisih_train_test"] > 0.05]
print(f"max_depth terbaik (CV F1-macro): {DEPTH_TERBAIK}")
if len(awal_overfit):
    print("Selisih train-test mulai > 5 poin pada max_depth =", awal_overfit.iloc[0]["max_depth"])

## 8. Skenario C: Penanganan Ketidakseimbangan Kelas (`class_weight`)
Kelas **Enrolled** paling sedikit. `class_weight="balanced"` memberi bobot lebih besar pada kelas minoritas.
Yang diamati: **recall kelas Enrolled** dan **F1-macro**.

In [ ]:
PILIHAN_CW = [None, "balanced", "balanced_subsample"]
baris = []
for cw in PILIHAN_CW:
    h, _ = jalankan({"n_estimators": N_TERBAIK, "max_depth": DEPTH_TERBAIK, "class_weight": cw}, X_train, X_test)
    baris.append({"class_weight": "Tanpa bobot" if cw is None else cw, "_nilai": cw, **h})
df_c = pd.DataFrame(baris)
display(df_c.drop(columns="_nilai").round(4))

fig, ax = plt.subplots(figsize=(8, 4.4))
lebar, pos = 0.26, np.arange(len(df_c))
ax.bar(pos - lebar, df_c["acc_test"], lebar, color=HIJAU_PASTEL, edgecolor=HIJAU_TUA, label="Akurasi testing")
ax.bar(pos, df_c["f1_test"], lebar, color=HIJAU, edgecolor=HIJAU_TUA, label="F1-macro testing")
ax.bar(pos + lebar, df_c["recall_enrolled"], lebar, color=WARNA_KELAS["Enrolled"], edgecolor=HIJAU_TUA, label="Recall kelas Enrolled")
ax.set_xticks(pos); ax.set_xticklabels(df_c["class_weight"]); ax.set_ylim(0, 1)
ax.set_title("Skenario C: Pengaruh Bobot Kelas"); ax.legend(loc="lower right")
simpan_grafik("skenario_c")

CW_TERBAIK = df_c.sort_values("cv_f1", ascending=False, kind="stable").iloc[0]["_nilai"]
print("class_weight terbaik (CV F1-macro):", CW_TERBAIK)
PARAM_TERBAIK = {"n_estimators": N_TERBAIK, "max_depth": DEPTH_TERBAIK, "class_weight": CW_TERBAIK}
print("Parameter terbaik:", PARAM_TERBAIK)

## 9. Skenario D: Prediksi Dini (Kelompok Fitur)
Seberapa baik model jika hanya memakai informasi yang tersedia **lebih awal**?

1. Hanya data pendaftaran (sebelum kuliah dimulai)
2. Data pendaftaran + hasil semester 1
3. Semua fitur (+ hasil semester 2)

In [ ]:
f_sem1 = [c for c in X.columns if c.startswith("Curricular units 1st")]
f_sem2 = [c for c in X.columns if c.startswith("Curricular units 2nd")]
f_daftar = [c for c in X.columns if c not in f_sem1 + f_sem2]
KELOMPOK = {
    "Pendaftaran saja": f_daftar,
    "Pendaftaran + Semester 1": f_daftar + f_sem1,
    "Semua fitur (+ Semester 2)": f_daftar + f_sem1 + f_sem2,
}
baris = []
for nama, kolom in KELOMPOK.items():
    h, _ = jalankan(PARAM_TERBAIK, X_train[kolom], X_test[kolom])
    baris.append({"kelompok": nama, "jumlah_fitur": len(kolom), **h})
df_d = pd.DataFrame(baris)
display(df_d.round(4))

fig, ax = plt.subplots(figsize=(8.4, 4.4))
lebar, pos = 0.34, np.arange(len(df_d))
ax.bar(pos - lebar / 2, df_d["acc_test"], lebar, color=HIJAU_PASTEL, edgecolor=HIJAU_TUA, label="Akurasi testing")
ax.bar(pos + lebar / 2, df_d["f1_test"], lebar, color=HIJAU, edgecolor=HIJAU_TUA, label="F1-macro testing")
for i, r in df_d.iterrows():
    ax.text(i - lebar / 2, r["acc_test"] + 0.01, f"{r['acc_test']:.2f}", ha="center", fontsize=9)
    ax.text(i + lebar / 2, r["f1_test"] + 0.01, f"{r['f1_test']:.2f}", ha="center", fontsize=9)
ax.set_xticks(pos); ax.set_xticklabels(df_d["kelompok"]); ax.set_ylim(0, 1)
ax.set_title("Skenario D: Prediksi Dini Berdasarkan Kelompok Fitur"); ax.legend(loc="lower right")
simpan_grafik("skenario_d")

## 10. Feature Importance
Dua sudut pandang: **importance bawaan Random Forest** (penurunan impurity, cepat tetapi cenderung condong ke fitur dengan banyak nilai)
dan **permutation importance** pada data uji (lebih jujur terhadap kinerja nyata).

In [ ]:
hasil_penuh, model_penuh = jalankan(PARAM_TERBAIK, X_train, X_test)
print("Model penuh (36 fitur):", {k: round(v, 4) for k, v in hasil_penuh.items()})

imp_gini = pd.Series(model_penuh.feature_importances_, index=X.columns).sort_values(ascending=False)
perm = permutation_importance(model_penuh, X_test, y_test, n_repeats=5, scoring="f1_macro",
                              random_state=RANDOM_STATE, n_jobs=-1)
imp_perm = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, serie, judul, warna in [(axes[0], imp_gini, "Importance bawaan (10 teratas)", HIJAU),
                                (axes[1], imp_perm, "Permutation importance (10 teratas)", KREM_TUA)]:
    atas = serie.head(10)[::-1]
    ax.barh(atas.index, atas.values, color=warna, edgecolor=HIJAU_TUA)
    ax.set_title(judul); ax.grid(axis="y", visible=False)
simpan_grafik("importance_top10")
display(pd.DataFrame({"importance_bawaan": imp_gini, "permutation": imp_perm}).round(4).head(15))

## 11. Seleksi Fitur untuk Aplikasi Web

Agar formulir web ringkas dan etis, tidak semua fitur dipakai:

* **Atribut sensitif dikeluarkan** (jenis kelamin, status pernikahan, internasional, kebutuhan khusus, status perantau). Model risiko tidak sebaiknya memperlakukan mahasiswa berbeda karena atribut pribadi.
  Ubah `KECUALIKAN_SENSITIF = False` jika ingin menyertakannya.
* **Fitur yang sulit diisi pengguna dikeluarkan**: pekerjaan/pendidikan orang tua, kewarganegaraan, dan indikator makro (pengangguran, inflasi, PDB).
* Dari kandidat yang tersisa, dipilih **`JUMLAH_FITUR_WEB` fitur terpenting** berdasarkan importance model penuh.

In [ ]:
KECUALIKAN_SENSITIF = True
JUMLAH_FITUR_WEB = 15

SENSITIF = ["Gender", "Marital status", "International", "Educational special needs", "Displaced"]
SULIT_DIISI = ["Nacionality", "Mother's qualification", "Father's qualification",
               "Mother's occupation", "Father's occupation",
               "Unemployment rate", "Inflation rate", "GDP"]

dikecualikan = SULIT_DIISI + (SENSITIF if KECUALIKAN_SENSITIF else [])
kandidat = [c for c in X.columns if c not in dikecualikan]
urut_kandidat = imp_gini[kandidat].sort_values(ascending=False)   # urutkan dari yang terpenting
terpilih = urut_kandidat.head(JUMLAH_FITUR_WEB).index.tolist()
FITUR_WEB = [c for c in X.columns if c in terpilih]          # urutan sesuai kolom asli

print(f"{len(kandidat)} kandidat -> {len(FITUR_WEB)} fitur dipakai di web:")
for i, c in enumerate(urut_kandidat.head(JUMLAH_FITUR_WEB).index, 1):
    print(f"{i:>2}. {c:<48} importance={imp_gini[c]:.4f}")

## 12. Model Akhir dan Evaluasi

In [ ]:
model_akhir = buat_model(**PARAM_TERBAIK).fit(X_train[FITUR_WEB], y_train)
pred = model_akhir.predict(X_test[FITUR_WEB])
proba = model_akhir.predict_proba(X_test[FITUR_WEB])

acc_train = accuracy_score(y_train, model_akhir.predict(X_train[FITUR_WEB]))
acc_test = accuracy_score(y_test, pred)
f1_macro = f1_score(y_test, pred, average="macro")
cv_akhir = cross_val_score(buat_model(**PARAM_TERBAIK), X_train[FITUR_WEB], y_train,
                           cv=SKF, scoring="f1_macro", n_jobs=-1).mean()

print(f"Akurasi training : {acc_train:.4f}")
print(f"Akurasi testing  : {acc_test:.4f}")
print(f"F1-macro testing : {f1_macro:.4f}")
print(f"F1-macro CV (5x) : {cv_akhir:.4f}")
print(f"Pembanding model 36 fitur -> akurasi {hasil_penuh['acc_test']:.4f}, F1-macro {hasil_penuh['f1_test']:.4f}\n")
print(classification_report(y_test, pred, target_names=KELAS, zero_division=0))

fig, ax = plt.subplots(figsize=(5.6, 4.8))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=KELAS, cmap=CMAP_HIJAU,
                                        values_format="d", ax=ax, colorbar=False)
ax.grid(False); ax.set_title("Confusion Matrix (Data Uji)")
ax.set_xlabel("Prediksi"); ax.set_ylabel("Sebenarnya")
simpan_grafik("confusion_matrix")

## 13. Menyimpan Model dan Artefak untuk Aplikasi Web
`app.py` hanya membaca file-file ini, sehingga angka di artikel, notebook, dan web selalu konsisten.

In [ ]:
# 1) Model (dikompresi agar muat di GitHub)
joblib.dump(model_akhir, MODEL_DIR / "rf_model.joblib", compress=3)
ukuran_mb = (MODEL_DIR / "rf_model.joblib").stat().st_size / 1_048_576
print(f"rf_model.joblib: {ukuran_mb:.1f} MB (batas file GitHub: 100 MB)")

# 2) Fitur + nilai referensi (dipakai web untuk penjelasan faktor)
Xw = X_train[FITUR_WEB]
referensi = {c: {"median": float(Xw[c].median()), "modus": float(Xw[c].mode().iloc[0]),
                 "min": float(Xw[c].min()), "max": float(Xw[c].max())} for c in FITUR_WEB}
tulis_json(MODEL_DIR / "fitur.json", {
    "fitur": FITUR_WEB, "referensi": referensi,
    "dikecualikan": {"sensitif": SENSITIF if KECUALIKAN_SENSITIF else [], "sulit_diisi": SULIT_DIISI},
})

# 3) Peta label
tulis_json(MODEL_DIR / "label_map.json", {str(k): v for k, v in label_map.items()})

# 4) Contoh data: satu mahasiswa uji per kelas (diprediksi benar dengan keyakinan tertinggi)
contoh = {}
Xt = X_test[FITUR_WEB].reset_index(drop=True)
for k, nama in label_map.items():
    cocok = np.where((y_test.values == k) & (pred == k))[0]
    if len(cocok):
        i = cocok[np.argmax(proba[cocok, k])]
        contoh[nama] = Xt.iloc[[i]].to_dict("records")[0]
tulis_json(MODEL_DIR / "contoh.json", contoh)

# 5) Importance
tulis_json(MODEL_DIR / "feature_importance.json", {
    "semua_fitur_bawaan": imp_gini.round(5).to_dict(),
    "semua_fitur_permutasi": imp_perm.round(5).to_dict(),
    "fitur_web": {c: float(imp_gini[c]) for c in sorted(FITUR_WEB, key=lambda c: -imp_gini[c])},
})

# 6) Metrik untuk halaman "Tentang Model"
laporan = classification_report(y_test, pred, target_names=KELAS, output_dict=True, zero_division=0)
tulis_json(MODEL_DIR / "metrik.json", {
    "dataset": {"jumlah_data": len(df), "jumlah_fitur": X.shape[1],
                "distribusi": df[TARGET].value_counts().to_dict()},
    "split": {"latih": len(X_train), "uji": len(X_test)},
    "skenario_a": df_a.to_dict("records"),
    "skenario_b": df_b.drop(columns="_nilai").to_dict("records"),
    "skenario_c": df_c.drop(columns="_nilai").to_dict("records"),
    "skenario_d": df_d.to_dict("records"),
    "parameter_terbaik": PARAM_TERBAIK,
    "model_penuh": {"jumlah_fitur": X.shape[1], "acc_test": hasil_penuh["acc_test"], "f1_test": hasil_penuh["f1_test"]},
    "model_akhir": {
        "jumlah_fitur": len(FITUR_WEB), "fitur": FITUR_WEB,
        "acc_train": acc_train, "acc_test": acc_test, "f1_macro": f1_macro, "cv_f1": cv_akhir,
        "laporan": {k: {m: laporan[k][m] for m in ("precision", "recall", "f1-score", "support")} for k in KELAS},
        "matriks": pd.crosstab(y_test.map(label_map), pd.Series(pred, index=y_test.index).map(label_map))
                     .reindex(index=KELAS, columns=KELAS, fill_value=0).values.tolist(),
        "kelas": KELAS,
    },
    "versi": {"sklearn": sklearn.__version__, "python": platform.python_version()},
})

# 7) requirements.txt (scikit-learn dikunci pada versi yang dipakai melatih model)
def v(paket, minimal=None):
    try:
        return f"{paket}=={version(paket)}"
    except PackageNotFoundError:
        return f"{paket}>={minimal}" if minimal else paket
(BASE_DIR / "requirements.txt").write_text("\n".join([
    v("Flask", "3.0"), "gunicorn", "pandas", "numpy", "joblib", f"scikit-learn=={sklearn.__version__}",
]) + "\n", encoding="utf-8")

print("Tersimpan di model/ :", sorted(p.name for p in MODEL_DIR.iterdir()))
print("Grafik di static/charts/ :", sorted(p.name for p in CHART_DIR.iterdir()))
print("\nrequirements.txt:\n" + (BASE_DIR / "requirements.txt").read_text())

## 14. Uji Cepat: Model Dimuat Ulang dan Memprediksi Contoh

In [ ]:
model_uji = joblib.load(MODEL_DIR / "rf_model.joblib")
for nama, baris in contoh.items():
    p = model_uji.predict_proba(pd.DataFrame([baris], columns=FITUR_WEB))[0]
    print(f"Contoh '{nama}': " + ", ".join(f"{label_map[int(c)]}={x*100:.1f}%" for c, x in zip(model_uji.classes_, p)))

## 15. Ringkasan untuk Artikel

| Bagian artikel | Isi dari notebook |
|---|---|
| Metode: data | Bagian 2-4, `distribusi_kelas.png` |
| Hasil: eksperimen | `skenario_a.png`, `skenario_b.png`, `skenario_c.png`, `skenario_d.png` |
| Hasil: fitur penting | `importance_top10.png` + daftar fitur terpilih (Bagian 11) |
| Hasil: evaluasi akhir | `confusion_matrix.png` + classification report (Bagian 12) |
| Pembahasan | Hubungkan Skenario D (prediksi dini) dan recall kelas Enrolled dengan keterbatasan model |

**Selanjutnya:** jalankan sel berikut (khusus Colab) untuk mengunduh paket hasil, lalu salin isinya ke folder proyek VS Code.

In [ ]:
if IN_COLAB:
    from google.colab import files
    paket = Path("/content/paket_hasil")
    shutil.rmtree(paket, ignore_errors=True)
    (paket / "static").mkdir(parents=True)
    shutil.copytree(MODEL_DIR, paket / "model")
    shutil.copytree(CHART_DIR, paket / "static" / "charts")
    shutil.copy(BASE_DIR / "requirements.txt", paket / "requirements.txt")
    shutil.make_archive("/content/hasil_notebook", "zip", paket)
    files.download("/content/hasil_notebook.zip")
else:
    print("Bukan Colab: semua file sudah tersimpan langsung di folder proyek.")